# Semantic pipeline: n_components and embedding model

**Purpose:** derive the semantic pipeline's remaining config constants -- `n_components`
and embedding model -- using the metric (c_v) and k-selection strategy (`find_best_cv_k`)
settled in notebooks 4-6.

**Reduction method: PCA.** Settled on methodological grounds, not an empirical SVD-vs-PCA
comparison: SVD was chosen for the lexical pipeline specifically because it avoids
centering sparse, non-negative count data -- a reason that doesn't apply to dense sentence
embeddings, where centering is the standard, textbook-correct choice. All reduction below
uses PCA.

**Order:** n_components is swept first (on the default embedding model), then the
embedding model comparison is measured at that settled n_components -- not at an arbitrary
earlier value -- so the two decisions don't confound each other.


In [1]:
import json
import sys
import time

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

sys.path.insert(0, ".")
from semantic_diagnostics import embed, coherence_scores, find_best_cv_k

with open("../data/stackoverflow_sample_2021_03_01_n5000.json") as f:
    data = json.load(f)
titles = pd.Series([row["title"] for row in data]).reset_index(drop=True)
print(f"{len(titles)} titles")


/home/olga/Documents/personal/clustering-json-fields/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5000 titles


## n_components (PCA, `all-MiniLM-L6-v2`)

Cumulative explained variance is reported alongside c_v specifically so any
information-loss gaming (inflating the score by discarding signal) would be visible
rather than assumed absent.

In [2]:
default_model = SentenceTransformer("all-MiniLM-L6-v2")
t0 = time.perf_counter()
minilm_embeddings = embed(titles, default_model)
minilm_embed_time = time.perf_counter() - t0

N_COMPONENTS_VALUES = [3, 5, 8, 10, 15, 20, 30, 50]
rows = []
for n_comp in N_COMPONENTS_VALUES:
    pca = PCA(n_components=n_comp, random_state=42)
    reduced = pca.fit_transform(minilm_embeddings)
    cum_var = pca.explained_variance_ratio_.sum()
    k, fits = find_best_cv_k(titles, reduced)
    labels = KMeans(n_clusters=k, init="k-means++", n_init=10, max_iter=100, random_state=42).fit_predict(reduced)
    c_v = coherence_scores(titles, labels, top_n=10)["c_v"]
    rows.append({"n_components": n_comp, "cum_variance": round(cum_var, 3), "search_k": k, "search_fits": fits, "c_v": round(c_v, 4)})

n_components_df = pd.DataFrame(rows)
n_components_df


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9003.84it/s]

,n_components,cum_variance,search_k,search_fits,c_v
0,3,0.084,142,23,0.3796
1,5,0.124,8,22,0.3858
2,8,0.178,7,22,0.3755
3,10,0.207,8,22,0.3982
4,15,0.270,22,23,0.3678
5,20,0.322,8,22,0.4083
6,30,0.403,13,22,0.3758
7,50,0.523,15,22,0.3946


## Embedding model, measured at the settled n_components

`all-MiniLM-L6-v2` (384-dim, small, fast) vs. `all-mpnet-base-v2` (768-dim, larger,
generally higher quality on standard embedding benchmarks), both reduced with PCA at the
n_components chosen above -- not a placeholder value -- so the quoted tradeoff is accurate
for the config actually being considered.

In [3]:
CHOSEN_N_COMPONENTS = int(n_components_df.loc[n_components_df["c_v"].idxmax(), "n_components"])
print(f"chosen n_components: {CHOSEN_N_COMPONENTS}")

mpnet_model = SentenceTransformer("all-mpnet-base-v2")
t0 = time.perf_counter()
mpnet_embeddings = embed(titles, mpnet_model)
mpnet_embed_time = time.perf_counter() - t0

model_results = []
for name, emb, embed_time in [
    ("all-MiniLM-L6-v2", minilm_embeddings, minilm_embed_time),
    ("all-mpnet-base-v2", mpnet_embeddings, mpnet_embed_time),
]:
    reduced = PCA(n_components=CHOSEN_N_COMPONENTS, random_state=42).fit_transform(emb)
    k, fits = find_best_cv_k(titles, reduced)
    labels = KMeans(n_clusters=k, init="k-means++", n_init=10, max_iter=100, random_state=42).fit_predict(reduced)
    c_v = coherence_scores(titles, labels, top_n=10)["c_v"]

    model_results.append({
        "model": name, "dim": emb.shape[1], "embed_s": round(embed_time, 1),
        "search_k": k, "search_fits": fits, "c_v": round(c_v, 4),
    })

pd.DataFrame(model_results)


chosen n_components: 20


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8686.75it/s]

,model,dim,embed_s,search_k,search_fits,c_v
0,all-MiniLM-L6-v2,384,10.9,8,22,0.4083
1,all-mpnet-base-v2,768,96.5,8,22,0.3815


## Observations

**n_components=20 gives the highest c_v in the sweep (0.4083), with a normal search_k
(8) -- not an edge case.** The range isn't flat this time: 0.3678 at n_components=15 up to
0.4083 at 20, a real spread rather than noise. n_components=3 stands out for an unusually
large search_k (142) despite unremarkable c_v (0.3796) -- a review-usability concern, not
a quality signal, consistent with the same pattern seen at very low dimensionality before.

**Measured at the actual n_components being used (20), `all-MiniLM-L6-v2` doesn't just
cost less than `all-mpnet-base-v2` -- it scores higher too** (0.4083 vs. 0.3815), on top of
embedding ~9x faster (10.9s vs. 96.5s for 5000 titles). The earlier placeholder comparison
(at a fixed n_components that happened to be a weak point for MiniLM specifically) made
this look like a real cost/quality tradeoff; once both are compared at the config actually
being committed, there's no tradeoff at all.

## Conclusion

- **Reduction method: PCA.** Centering is the textbook-correct choice for dense sentence
  embeddings; SVD's centering-avoidance only mattered for the lexical path's sparse count
  data.
- **n_components: 20.** Best c_v in the sweep, with a normal, reviewable cluster count.
- **Embedding model: `all-MiniLM-L6-v2`.** Faster and higher-scoring than
  `all-mpnet-base-v2` at the settled n_components -- an easy call, not a tradeoff.
